# Week 22: ML for satellite imagery: CNNs and U-Net segmentation

**Track:** Space GIS Architect (Expert)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/22/](https://launchdetect.com/academy/week/22/)  
**Track index:** [https://launchdetect.com/academy/space-gis-architect/](https://launchdetect.com/academy/space-gis-architect/)

---

_Deep learning has rewritten remote sensing. CNNs (object detection) and U-Nets (semantic segmentation) are now standard. This week you train one on real GOES data._


## Why this week matters

Deep learning has replaced threshold rules for most production segmentation tasks. A trained U-Net catches plume edges that a simple threshold misses, and produces probabilistic output for downstream confidence scoring.


## Learning objectives

By the end of this lab you will be able to:

- Train a CNN for object detection in raster imagery
- Build a U-Net for semantic segmentation of clouds / plumes / fires
- Generate training data via thresholding + manual labels
- Evaluate with IoU and confusion matrices


## Setup — and why these dependencies

This lab uses: `leafmap[common] xarray rasterio pystac-client boto3`. Run the cell below once; Colab installs into the runtime.


In [ ]:
# Install required packages
!pip install -q leafmap[common] xarray rasterio pystac-client boto3


## The approach (and why)

Build a minimal U-Net in PyTorch. Use weakly-supervised training data (Week 14 threshold detections + morphology cleanup as 'labels'). Evaluate with IoU.


In [ ]:
# Week 22: simple U-Net scaffold for plume segmentation.
import torch, torch.nn as nn

class TinyUNet(nn.Module):
    """Minimal U-Net — encoder-decoder + 1 skip connection."""
    def __init__(self, in_ch=1, out_ch=1, base=16):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(in_ch, base, 3, padding=1), nn.ReLU(),
                                   nn.Conv2d(base, base, 3, padding=1), nn.ReLU())
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = nn.Sequential(nn.Conv2d(base, base*2, 3, padding=1), nn.ReLU(),
                                         nn.Conv2d(base*2, base*2, 3, padding=1), nn.ReLU())
        self.up = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(base*2, base, 3, padding=1), nn.ReLU(),
                                   nn.Conv2d(base, out_ch, 1))

    def forward(self, x):
        e1 = self.enc1(x)
        b = self.bottleneck(self.pool(e1))
        d1 = self.up(b)
        return self.dec1(torch.cat([d1, e1], dim=1))

model = TinyUNet()
x = torch.randn(1, 1, 64, 64)
y = model(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {y.shape}  (per-pixel logits)")
print(f"Params:       {sum(p.numel() for p in model.parameters()):,}")

# TODO: generate weak-supervised plume masks from Week 14 threshold detection,
# train this U-Net, evaluate with IoU. Confirm FPR < 5%.


## What just happened — and why it works

U-Net is an encoder-decoder with skip connections. The encoder downsamples to extract abstract features; the decoder upsamples back to original resolution; skip connections preserve fine spatial detail that the bottleneck would otherwise lose. The output is a per-pixel class probability.


## Common gotchas

- Class imbalance is severe (99.99% of pixels are not plumes). Use weighted loss (BCE with pos_weight, or Dice loss).
- Weak supervision is noisy — accept it and train through it. The network learns the spatial context the rules can't.
- Don't overfit to small training sets. Start with a tiny U-Net (1M params) and grow only if validation IoU plateaus.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/22/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
